<center><img src="../logo.png" alt="Header" style="width: 800px;"/></center>

In [ ]:
#!/usr/bin/env python
# coding: utf-8

import cv2
import json
import threading
import time
import requests
from picamera2 import Picamera2, Preview
import libcamera
import smbus
import math
from flask import Flask, request
import threading


class YB_Pcb_Car(object):
    def get_i2c_device(self, address, i2c_bus):
        self._addr = address
        if i2c_bus is None:
            return smbus.SMBus(1)
        else:
            return smbus.SMBus(i2c_bus)

    def __init__(self):
        # 创建 I2C 设备
        try:
            self._device = self.get_i2c_device(0x16, 1)
            print("电机控制器初始化成功")
        except Exception as e:
            print(f"初始化失败: {str(e)}")
            self._device = None

    def write_u8(self, reg, data):
        try:
            self._device.write_byte_data(self._addr, reg, data)
        except Exception as e:
            print(f'write_u8 I2C error: {e}')

    def write_reg(self, reg):
        try:
            self._device.write_byte(self._addr, reg)
        except Exception as e:
            print(f'write_reg I2C error: {e}')

    def write_array(self, reg, data):
        try:
            self._device.write_i2c_block_data(self._addr, reg, data)
        except Exception as e:
            print(f'write_array I2C error: {e}')

    def Ctrl_Car(self, l_dir, l_speed, r_dir, r_speed):
        try:
            reg = 0x01
            data = [l_dir, l_speed, r_dir, r_speed]
            self.write_array(reg, data)
        except Exception as e:
            print(f'Ctrl_Car I2C error: {e}')
            
    def Control_Car(self, speed1, speed2):
        try:
            if speed1 < 0:
                dir1 = 0
            else:
                dir1 = 1
            if speed2 < 0:
                dir2 = 0
            else:
                dir2 = 1 
            self.Ctrl_Car(dir1, int(math.fabs(speed1)), dir2, int(math.fabs(speed2)))
        except Exception as e:
            print(f'Control_Car I2C error: {e}')

    def Car_Run(self, speed1, speed2):
        # 前进（正方向旋转）
        try:
            self.Ctrl_Car(1, speed1, 1, speed2)
        except Exception as e:
            print(f'Car_Run I2C error: {e}')

    def Car_Stop(self):
        # 急停，利用专用寄存器停止
        try:
            reg = 0x02
            self.write_u8(reg, 0x00)
        except Exception as e:
            print(f'Car_Stop I2C error: {e}')

    def Car_Back(self, speed1, speed2):
        # 后退（反方向旋转）
        try:
            self.Ctrl_Car(0, speed1, 0, speed2)
        except Exception as e:
            print(f'Car_Back I2C error: {e}')

    def Car_Left(self, speed1, speed2):
        # 左转：通常设置左电机反转，右电机正转
        try:
            self.Ctrl_Car(0, speed1, 1, speed2)
        except Exception as e:
            print(f'Car_Left I2C error: {e}')

    def Car_Right(self, speed1, speed2):
        # 右转：通常设置左电机正转，右电机反转
        try:
            self.Ctrl_Car(1, speed1, 0, speed2)
        except Exception as e:
            print(f'Car_Right I2C error: {e}')

    def Car_Spin_Left(self, speed1, speed2):
        # 原地左旋
        try:
            self.Ctrl_Car(0, speed1, 1, speed2)
        except Exception as e:
            print(f'Car_Spin_Left I2C error: {e}')

    def Car_Spin_Right(self, speed1, speed2):
        # 原地右旋
        try:
            self.Ctrl_Car(1, speed1, 0, speed2)
        except Exception as e:
            print(f'Car_Spin_Right I2C error: {e}')

    def Ctrl_Servo(self, servo_id, angle):
        # 控制舵机，确保角度在 0~180 范围内
        try:
            if angle < 0:
                angle = 0
            elif angle > 180:
                angle = 180
            reg = 0x03
            data = [servo_id, angle]
            self.write_array(reg, data)
        except Exception as e:
            print(f'Ctrl_Servo I2C error: {e}')


app = Flask(__name__)

class RobotController:
    def __init__(self):
        self.command = None   # 来自服务器的控制命令
        self.car = YB_Pcb_Car()   # 初始化官方完整版本的电机控制器
        self.running = True
        self.lock = threading.Lock()

        self.camera_thread = None  # 用来控制摄像头线程

    def image_worker(self):
        picam2 = Picamera2()
        config = picam2.create_preview_configuration(
            main={"format": 'RGB888', "size": (640, 480)}
        )
        config["transform"] = libcamera.Transform(hflip=1, vflip=1)
        
        try:
            picam2.configure(config)
            with picam2:
                picam2.start()
                print("摄像头已启动")
                while self.running:
                    frame = picam2.capture_array()
                    self.image_display(frame)
                    self.send_frame(frame)
                    time.sleep(0.01)
        except Exception as e:
            print(f"摄像头错误: {str(e)}")
        finally:
            try:
                picam2.stop()
            except Exception:
                pass

    def start_camera(self):
        if self.camera_thread is None or not self.camera_thread.is_alive():
            self.camera_thread = threading.Thread(target=self.image_worker, daemon=True)
            self.camera_thread.start()

    def stop_camera(self):
        self.running = False
        if self.camera_thread is not None and self.camera_thread.is_alive():
            self.camera_thread.join()

    @app.route('/start_camera', methods=['POST'])
    def start_camera_api(self):
        self.start_camera()
        return "摄像头已启动", 200

    @app.route('/stop_camera', methods=['POST'])
    def stop_camera_api(self):
        self.stop_camera()
        return "摄像头已停止", 200

    def motor_worker(self):
        CONTROL_INTERVAL = 0.1
        while self.running:
            with self.lock:
                cmd = self.command
                self.command = None

            if not cmd:
                time.sleep(CONTROL_INTERVAL)
                continue
            
            print(f"【执行指令】{cmd}")

            try:
                if cmd == "forward":
                    self.car.Car_Run(70, 70)
                elif cmd == "backward":
                    self.car.Car_Back(40, 40)
                elif cmd == "left":
                    self.car.Car_Left(50, 60)
                elif cmd == "right":
                    self.car.Car_Right(60, 50)
                elif cmd == "spin_left":
                    self.car.Car_Spin_Left(50, 50)
                elif cmd == "spin_right":
                    self.car.Car_Spin_Right(50, 50)
                elif cmd == "stop":
                    self.car.Car_Stop()
                elif cmd == "curve_left":
                    self.car.Control_Car(-20, 50)
                elif cmd == "curve_right":
                    self.car.Control_Car(50, -20)
                else:
                    print(f"【未识别指令】{cmd}")
            except Exception as e:
                print(f"电机控制异常: {e}")

            time.sleep(CONTROL_INTERVAL)

    def emergency_stop(self):
        self.running = False
        self.car.Car_Stop()

# 启动 Flask 应用
@app.before_first_request
def before_first_request():
    controller = RobotController()
    controller.start_camera()

if __name__ == '__main__':
    # 启动 Flask 服务器
    app.run(host='0.0.0.0', port=5000)
